### Discovering clusters with DatabionicSwarm

We walk through the algorithm in two acts:

1. **Toy dataset** — 9 points in 2D, three obvious clusters.  
   We visualise the raw data, inspect the distance matrix, run the swarm, and read the result.
2. **lsun benchmark** — 404 real data points with non-convex shapes.  
   Same pipeline, bigger data, compared against the published paper result.

In [1]:
include("/Users/mattijnvanhoek/mattijn/pyesom/notebooks/dbs.jl")
using .DatabionicSwarm
println("module loaded")

module loaded


## Part 1 — Toy dataset

Three clusters of 3 points each, arranged in 2D space so the eye can verify the answer.

We use `show_data` to draw a quick ASCII scatter plot, then `pairwise_euclidean` to build
the distance matrix, and `show_dist` to render it as a shade heatmap.

Because the clusters are well separated, the distance matrix should show a clear
**block structure**: light shade on the three diagonal blocks (within-cluster, small distances),
dark shade everywhere else (between clusters, large distances).

In [2]:
# Three clusters, 3 points each — well separated in 2D
X = Float64[
    1.0  4.0;   # A  (top-left)
    1.5  4.5;
    0.8  3.8;
    5.0  4.0;   # B  (top-right)
    5.5  4.3;
    4.8  3.7;
    3.0  0.5;   # C  (bottom-centre)
    3.3  0.8;
    2.8  0.3]
labels = ["A","A","A","B","B","B","C","C","C"]

show_data(X, labels; title="Toy dataset")
D = pairwise_euclidean(X)
show_dist(D, labels; title="Pairwise distances")

Toy dataset  (9 points)

  ┌────────────────────────────────────┐
  │     A                              │
  │                                   B│
  │AA                            BB    │
  │                                    │
  │                                    │
  │                                    │
  │                                    │
  │                                    │
  │                                    │
  │                                    │
  │                                    │
  │                   C                │
  │                C                   │
  │               C                    │
  └────────────────────────────────────┘

Pairwise distances  (9 × 9)

    A A A B B B C C C 
A     ·   ▓ █ ▓ ▓ ▓ ▓ 
A   ·   · ▓ ▓ ▓ ▓ ▓ █ 
A     ·   ▓ █ ▓ ▓ ▓ ▓ 
B   ▓ ▓ ▓   ·   ▓ ▓ ▓ 
B   █ ▓ █ ·   · █ ▓ █ 
B   ▓ ▓ ▓   ·   ▓ ▒ ▓ 
C   ▓ ▓ ▓ ▓ █ ▓       
C   ▓ ▓ ▓ ▓ ▓ ▒     · 
C   ▓ █ ▓ ▓ █ ▓   ·   



In short the algorithm does the following:
- define the number of bots, equalling the number of samples — for the toy example that is **9 bots**
- it creates a grid large enough for the bots to move around freely (area ≈ 3–5× the number of bots) — for 9 bots this becomes a **7 × 7 grid**

The following image shows what a random initial placement looks like on a slightly larger example (9×9 grid, 20 bots):

```
Grid 9×9  20 bots

 ●  ·  ·  ·  ●  ·  ·  ·  · 
 ·  ·  ·  ·  ·  ·  ·  ●  · 
 ·  ·  ·  ·  ·  ·  ●  ·  · 
 ●  ·  ·  ·  ·  ●  ·  ·  ● 
 ·  ●  ·  ·  ·  ●  ·  ·  ● 
 ·  ·  ·  ·  ·  ·  ·  ·  · 
 ●  ·  ●  ·  ·  ·  ·  ●  · 
 ·  ●  ·  ·  ·  ●  ·  ●  ● 
 ●  ·  ●  ·  ·  ·  ·  ●  · 
```

- based on the grid size, we define a maximum and minimum search radius **Rmax → Rmin**; at each step bots try to jump to ring positions at exactly that radius
- at high R many bots make wide jumps (exploration); at low R few bots make tiny adjustments (fine-tuning)
- movement is pac-man style: crossing an edge reappears on the opposite side

The following image shows potential positions of a single bot for 3 different radii and 2 different starting positions:

```
Grid 9×9  bot=(5,5)  R=1       Grid 9×9  bot=(5,5)  R=3      Grid 9×9  bot=(1,1)  R=2

 ·  ·  ·  ·  ·  ·  ·  ·  ·     ·  ·  ·  ·  ·  ·  ·  ·  ·     ●  ·  ○  ·  ·  ·  ·  ○  · 
 ·  ·  ·  ·  ·  ·  ·  ·  ·     ·  ·  ○  ○  ○  ○  ○  ·  ·     ·  ·  ○  ·  ·  ·  ·  ○  · 
 ·  ·  ·  ·  ·  ·  ·  ·  ·     ·  ○  ·  ·  ·  ·  ·  ○  ·     ○  ○  ○  ·  ·  ·  ·  ○  ○ 
 ·  ·  ·  ○  ○  ○  ·  ·  ·     ·  ○  ·  ·  ·  ·  ·  ○  ·     ·  ·  ·  ·  ·  ·  ·  ·  · 
 ·  ·  ·  ○  ●  ○  ·  ·  ·     ·  ○  ·  ·  ●  ·  ·  ○  ·     ·  ·  ·  ·  ·  ·  ·  ·  · 
 ·  ·  ·  ○  ○  ○  ·  ·  ·     ·  ○  ·  ·  ·  ·  ·  ○  ·     ·  ·  ·  ·  ·  ·  ·  ·  · 
 ·  ·  ·  ·  ·  ·  ·  ·  ·     ·  ○  ·  ·  ·  ·  ·  ○  ·     ·  ·  ·  ·  ·  ·  ·  ·  · 
 ·  ·  ·  ·  ·  ·  ·  ·  ·     ·  ·  ○  ○  ○  ○  ○  ·  ·     ○  ○  ○  ·  ·  ·  ·  ○  ○ 
 ·  ·  ·  ·  ·  ·  ·  ·  ·     ·  ·  ·  ·  ·  ·  ·  ·  ·     ·  ·  ○  ·  ·  ·  ○  ·  · 
```

---

After the bots have settled we want to know two things: **where** each bot ended up (the projection) and **how different** neighbouring bots are from each other.

`show_grid` prints both at once.  Each cell on the grid is either:
- a **bot label** — the data-point that landed there
- a **shade character** — the average pairwise data distance of all bots within a small radius

```
shade:  ' '  ·   ░   ▒   ▓   █
dist:   low ←——————————————→ high
```

Low shade between two bots means they are similar (same cluster). Dark shade (▓ █) separates bots that are far apart in the data — a **cluster boundary**.

In [3]:
result = pswarm!(D, seed=42)
show_grid(result; labels=labels)

Bots  : 9
Grid  : 7 × 7 = 49 cells
Rmax  : 3  Rmin : 1

R= 3  epochs= 10  stress=0.0431
R= 2  epochs= 10  stress=0.0431
R= 1  epochs= 10  stress=0.0376

Done. Final stress : 0.0376
Grid  7×7  (label=bot  ░▒▓█=cluster boundary)

 ▒  █              ▓ 
 A              █  ▓ 
 A              ░  ▒ 
 ▓  A     █     C  C 
 ▓  ▓           C  ▒ 
 B  ▒           ▓  ▒ 
 B              ▓  B 



Three bots labelled `A` settle in one region, `B` in another, `C` in a third.
Dark `█` cells trace the **boundaries** between clusters — cells whose surrounding bots have
high pairwise data distances.

The swarm had **no access to the raw coordinates** — only the 9×9 distance matrix.
It discovered the cluster structure purely from "how far apart is every pair of points."

In [4]:
show_stress(result.stress_history)

Stress History  (30 epochs)

0.1120  │•                                                  
        │  •                                                
        │   •                                               
        │                                                   
        │                                                   
        │     • • •• •                                      
0.0376  │              • •• • •• • • •• • •• • • •• • • •• •
        └───────────────────────────────────────────────────
        1                                                 30



`pswarm_live!` runs the same algorithm but redraws the grid and stress curve after every epoch,
so you can watch the bots self-organise in real time.

Early on (large R) the 9 bots are scattered randomly across the grid and stress is high.
As R decreases they begin to gravitate toward each other — same-cluster bots pull together,
different-cluster bots push apart.  By the final epochs the three groups are clearly separated
and the shade pattern has stabilised.

In [7]:
pswarm_live!(D; labels=labels, seed=42, fps=4)
println("live training finished")

R=1  epoch=10  stress=0.0376  jumps=0  7×7  (label=bot  ░▒▓█=cluster boundary)

 ▒  █              ▓ 
 A              █  ▓ 
 A              ░  ▒ 
 ▓  A     █     C  C 
 ▓  ▓           C  ▒ 
 B  ▒           ▓  ▒ 
 B              ▓  B 

Stress History  (30 epochs)

0.1120  │•                                                  
        │  •                                                
        │   •                                               
        │                                                   
        │                                                   
        │     • • •• •                                      
0.0376  │              • •• • •• • • •• • •• • • •• • • •• •
        └───────────────────────────────────────────────────
        1                                                 30


Done. Final stress: 0.0376
live training finished


The stress curve confirms convergence: a steep drop while bots explore at large radius,
then a flat tail once they are settled.

---

## Part 2 — lsun benchmark

**lsun** is a 2D benchmark from the FCPS suite with 404 data points in three non-trivial shapes:
a tall L-shaped region and two compact squares. Same pipeline, bigger data.

We z-score both axes so neither dominates the distance, then hand the full 404×404 distance matrix
to the swarm — **no coordinates, no labels, just distances**.

In [8]:
using DelimitedFiles

raw         = readdlm(joinpath(@__DIR__, "lsun.csv"), ',', skipstart=1)
X_lsun      = Float64.(raw[:, 1:2])
labels_lsun = string.(Int.(raw[:, 3]))

show_data(X_lsun, labels_lsun; title="lsun (raw coordinates)", width=50, height=18)

D_lsun = pairwise_euclidean(zscore(X_lsun))
println("lsun: $(size(X_lsun,1)) samples   classes: $(sort(unique(labels_lsun)))")
println("D:    $(size(D_lsun))   range $(round(minimum(filter(x->x>0, vec(D_lsun))), digits=3)) – $(round(maximum(D_lsun), digits=3))")

result_lsun = pswarm!(D_lsun, max_epochs=50)
show_grid(result_lsun; labels=labels_lsun)
show_stress(result_lsun.stress_history)

lsun (raw coordinates)  (404 points)

  ┌──────────────────────────────────────────────────┐
  │         1 11  1 1                                │
  │        1111 1  11                            33  │
  │       11111  111                            3    │
  │       111  1 11                                  │
  │       111 1 1  11                                │
  │        1 1 1 1 11                                │
  │      111    111            2         2           │
  │         11111        2      2 2 2 2 222 2        │
  │      1 11  1          2  2 2 2 22 2222     2     │
  │       1  1 11 11         2 222222222222  2   2   │
  │       1     1  1        2  2  22222222  2  2    2│
  │      1   1 111 1          2 22     22 2   2      │
  │      111     111                2                │
  │                                                  │
  │0 00 00 00 0000000000 0 00 0 0 0 0 0000   000     │
  │0 00 000000 000000 00000 0000  000 00 0 000000    │
  │000000 00000000 000 00 0

Labels are the true lsun cluster ids: **0** = L-shape (200 pts), **1** = top square (100 pts),
**2** = bottom square (100 pts), and **3** = 4 noise points near the boundaries.

If the algorithm worked, same-label bots land close together and dark `█` cells trace the
boundaries between the three non-convex regions — shapes that confound algorithms that assume
spherical clusters.

The stress curve shows the same pattern as the toy example, just with more epochs and a longer
settling tail because there are 404 bots on a 37×37 grid.